In [1]:
# ======================================
# LOANPALZ CREDIT RISK SYSTEM
# ======================================

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

RANDOM_STATE = 42

In [2]:
# =========================
# LOAD DATA
# =========================

train_df = pd.read_csv("../data/train.csv")
train_df = train_df.drop("Unnamed: 0", axis=1)

print("Dataset shape:", train_df.shape)
train_df.head()

Dataset shape: (150000, 11)


,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
1,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
2,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
3,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
4,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


In [3]:
# =========================
# MISSING VALUES
# =========================

train_df["MonthlyIncome"] = train_df["MonthlyIncome"].fillna(
    train_df["MonthlyIncome"].median()
)

train_df["NumberOfDependents"] = train_df["NumberOfDependents"].fillna(
    train_df["NumberOfDependents"].median()
)

print("Missing values handled")

Missing values handled


In [4]:
# =========================
# OUTLIER CLIPPING
# =========================

clip_cols = [
    "MonthlyIncome",
    "DebtRatio",
    "RevolvingUtilizationOfUnsecuredLines",
    "NumberOfOpenCreditLinesAndLoans",
    "NumberRealEstateLoansOrLines"
]

for col in clip_cols:
    lower = train_df[col].quantile(0.01)
    upper = train_df[col].quantile(0.99)
    train_df[col] = train_df[col].clip(lower, upper)

print("Outliers clipped")

Outliers clipped


In [5]:
# =========================
# MIN-MAX NORMALIZATION
# =========================

normalize_cols = [
    "MonthlyIncome",
    "NumberOfDependents",
    "NumberOfOpenCreditLinesAndLoans",
    "NumberRealEstateLoansOrLines",
    "age"
]

scaler_norm = MinMaxScaler()
train_df[normalize_cols] = scaler_norm.fit_transform(train_df[normalize_cols])

print("Normalization complete")

Normalization complete


In [6]:
# ======================================
# BEHAVIORAL CREDIT INDICES
# ======================================

# Financial Stability Index
train_df["FSI"] = (
    train_df["MonthlyIncome"]
    * (1 - train_df["DebtRatio"])
    * (1 / (1 + train_df["NumberOfDependents"]))
)

# Payment Discipline Score
train_df["PDS"] = 1 / (
    1
    + 0.2 * train_df["NumberOfTime30-59DaysPastDueNotWorse"]
    + 0.3 * train_df["NumberOfTime60-89DaysPastDueNotWorse"]
    + 0.5 * train_df["NumberOfTimes90DaysLate"]
)

# Total Loans
train_df["TotalLoans"] = (
    train_df["NumberOfOpenCreditLinesAndLoans"]
    + train_df["NumberRealEstateLoansOrLines"]
)

# Debt Stress Index
train_df["DSI"] = (
    train_df["DebtRatio"]
    + train_df["RevolvingUtilizationOfUnsecuredLines"]
) * np.log1p(train_df["TotalLoans"])

# Credit Utilization Risk Score
train_df["CURS"] = (
    train_df["RevolvingUtilizationOfUnsecuredLines"]
    / (1 + train_df["NumberOfOpenCreditLinesAndLoans"])
)

print("Behavioral features created")
train_df[["FSI","PDS","DSI","CURS"]].describe()

Behavioral features created


,FSI,PDS,DSI,CURS
count,150000.000000,150000.000000,150000.000000,150000.000000
mean,-68.479370,0.937025,177.862121,0.250070
std,203.950660,0.147070,597.404410,0.292883
min,-1168.757217,0.010101,0.000000,0.000000
25%,0.014243,1.000000,0.117683,0.022042
50%,0.118771,1.000000,0.325519,0.111568
75%,0.219652,1.000000,0.886866,0.405419
max,1.000000,1.000000,5471.014016,1.092956


In [9]:
# =========================
# FEATURES & TARGET
# =========================

X = train_df.drop("SeriousDlqin2yrs", axis=1)
y = train_df["SeriousDlqin2yrs"]

print("Feature shape:", X.shape)

Feature shape: (150000, 15)


In [10]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)

Train: (120000, 15)
Validation: (30000, 15)


In [11]:
# =========================
# FINAL MODEL TRAINING
# =========================

xgb_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    eval_metric="logloss",
    random_state=RANDOM_STATE
)

xgb_model.fit(X_train_bal, y_train_bal)

print("Model trained")

NameError: name 'X_train_bal' is not defined